In [1]:
import edsnlp

# Création du pipeline
nlp = edsnlp.blank("fr")
nlp.add_pipe("eds.sentences")
nlp.add_pipe("eds.normalizer")
nlp.add_pipe("eds.dates")
nlp.add_pipe("eds.matcher", config={
    "terms": {
        "cardiologie": ["douleurs thoraciques", "insuffisance cardiaque", "ECG"],
        "neurologie": ["migraine", "AVC", "épilepsie", "céphalées"],
        "pneumologie": ["toux", "dyspnée", "asthme", "bronchite"],
        "symptomes": ["douleur", "fièvre", "fatigue", "nausée"]
    }
})

print("✅ Pipeline EDS-NLP chargé avec succès !")


/Users/nourhtekam/Documents/M2-E3IN BIG Data & IA/Projet LAB/Ctd-IA/ctd-ia/venv/lib/python3.14/site-packages/confit/errors.py:18: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.error_wrappers import (


✅ Pipeline EDS-NLP chargé avec succès !


In [2]:
# Texte médical de test
texte = """
Le patient Jean Dupont, né le 12/03/1965, consulte le 20/01/2024 
pour des douleurs thoraciques depuis 3 jours. 
Il présente également de la fièvre à 38.5°C.
Antécédents : insuffisance cardiaque diagnostiquée en 2020.
"""

# On analyse le texte avec notre pipeline
doc = nlp(texte)

# On affiche les entités détectées
print("📋 Entités détectées :")
for ent in doc.ents:
    print(f"  → '{ent.text}' | Type : {ent.label_}")


📋 Entités détectées :
  → 'douleurs thoraciques' | Type : cardiologie
  → 'fièvre' | Type : symptomes
  → 'insuffisance cardiaque' | Type : cardiologie


In [6]:
import spacy
import re

# Chargement du modèle français
nlp_spacy = spacy.load("fr_core_news_sm")

def anonymiser(texte: str) -> str:
    """Anonymise un texte médical avec spaCy + regex"""
    
    doc = nlp_spacy(texte)
    texte_anonyme = texte
    
    # Remplace les personnes et lieux détectés par spaCy
    for ent in reversed(doc.ents):
        if ent.label_ == "PER":
            texte_anonyme = texte_anonyme.replace(ent.text, "[PATIENT]")
        elif ent.label_ == "LOC":
            texte_anonyme = texte_anonyme.replace(ent.text, "[LIEU]")
        elif ent.label_ == "ORG":
            texte_anonyme = texte_anonyme.replace(ent.text, "[ORGANISATION]")
    
    # Remplace les dates avec regex
    texte_anonyme = re.sub(r'\d{2}/\d{2}/\d{4}', '[DATE]', texte_anonyme)
    texte_anonyme = re.sub(r'\d{2}/\d{4}', '[DATE]', texte_anonyme)
    
    # Anonymiser les années seules (ex: 2020, 1998...)
    texte_anonyme = re.sub(r'\b(19|20)\d{2}\b', '[DATE]', texte_anonyme)

    return texte_anonyme

# Test
print("🔒 Texte anonymisé :")
print(anonymiser(texte))


🔒 Texte anonymisé :

Le patient [PATIENT], né le [DATE], consulte le [DATE] 
pour des douleurs thoraciques depuis 3 jours. 
Il présente également de la fièvre à 38.5°C.
[PATIENT] : insuffisance cardiaque diagnostiquée en [DATE].



##  Enrichissement des dictionnaires 
Ajout des topics rythmologiques (Brady, Tachy, CRT, EP, ICM, PFA, CSP) et des hashtags associés.

In [2]:
# ============================================================
#  DICTIONNAIRE TOPICS RYTHMOLOGIQUES
#  Brady, Tachy, CRT, EP, ICM, PFA, CSP
# ============================================================

topics_keywords = {
    "Brady": [
        "bradycardie", "bradyarythmie",
        "bloc auriculo-ventriculaire", "BAV", "BAV 1", "BAV 2", "BAV 3",
        "bloc de branche", "dysfonction sinusale", "maladie de l'oreillette",
        "pause sinusale", "stimulateur cardiaque", "pacemaker", "PM",
        "implantation pacemaker", "sonde de stimulation"
    ],
    "Tachy": [
        "tachycardie", "tachyarythmie",
        "fibrillation auriculaire", "FA", "flutter auriculaire",
        "tachycardie supraventriculaire", "TSV",
        "tachycardie ventriculaire", "TV", "fibrillation ventriculaire", "FV",
        "wolff-parkinson-white", "WPW", "syndrome de Brugada",
        "torsades de pointes"
    ],
    "CRT": [
        "resynchronisation cardiaque", "CRT", "CRT-P", "CRT-D",
        "défibrillateur biventriculaire", "biventriculaire",
        "insuffisance cardiaque avancée", "FEVG basse",
        "sonde ventriculaire gauche", "sinus coronaire"
    ],
    "EP": [
        "électrophysiologie", "exploration électrophysiologique", "EEP",
        "ablation", "ablation par radiofréquence", "ablation par cathéter",
        "mapping", "cartographie", "cathéter",
        "isolation des veines pulmonaires", "IVP",
        "ablation flutter", "ablation FA", "ablation TV"
    ],
    "ICM": [
        "moniteur cardiaque implantable", "ICM", "Holter implantable",
        "Reveal", "enregistreur cardiaque", "monitoring longue durée",
        "détection arythmie", "bilan syncope", "syncope inexpliquée"
    ],
    "PFA": [
        "électroporation", "PFA", "pulsed field ablation",
        "champ électrique pulsé", "ablation par champ pulsé",
        "FARAPULSE", "ablation non thermique"
    ],
    "CSP": [
        "conduction His", "pacing His", "His bundle pacing",
        "LBBAP", "left bundle branch area pacing",
        "pacing septal gauche", "stimulation physiologique",
        "His-Purkinje", "branche gauche"
    ]
}

print("✅ Dictionnaire topics_keywords chargé !")
for topic, kws in topics_keywords.items():
    print(f"  → {topic} : {len(kws)} mots-clés")

✅ Dictionnaire topics_keywords chargé !
  → Brady : 16 mots-clés
  → Tachy : 15 mots-clés
  → CRT : 10 mots-clés
  → EP : 14 mots-clés
  → ICM : 9 mots-clés
  → PFA : 7 mots-clés
  → CSP : 9 mots-clés


In [3]:
# ============================================================
#  DICTIONNAIRE HASHTAGS RYTHMOLOGIQUES
# ============================================================

hashtags_keywords = {
    "Brady": [
        "#bradycardie", "#pacemaker", "#BAV", "#stimulateurCardiaque",
        "#blocDebranche", "#dysfonctionSinusale"
    ],
    "Tachy": [
        "#tachycardie", "#fibrillatonAuriculaire", "#FA", "#flutter",
        "#TSV", "#tachycardieVentriculaire", "#WPW", "#Brugada"
    ],
    "CRT": [
        "#CRT", "#resynchronisation", "#CRTd", "#CRTp",
        "#biventriculaire", "#insuffisanceCardiaque"
    ],
    "EP": [
        "#ablation", "#electrophysiologie", "#mapping", "#catheter",
        "#isolationVeinesPulmonaires", "#ablationFA", "#ablationTV"
    ],
    "ICM": [
        "#moniteurImplantable", "#ICM", "#HolterImplantable",
        "#syncope", "#monitoringCardiaque"
    ],
    "PFA": [
        "#PFA", "#electroporation", "#pulsedFieldAblation",
        "#ablationNonThermique", "#FARAPULSE"
    ],
    "CSP": [
        "#LBBAP", "#HisPacing", "#stimulationPhysiologique",
        "#conductionHis", "#brancheGauche"
    ]
}

print("✅ Dictionnaire hashtags_keywords chargé !")
for topic, tags in hashtags_keywords.items():
    print(f"  → {topic} : {tags}")

✅ Dictionnaire hashtags_keywords chargé !
  → Brady : ['#bradycardie', '#pacemaker', '#BAV', '#stimulateurCardiaque', '#blocDebranche', '#dysfonctionSinusale']
  → Tachy : ['#tachycardie', '#fibrillatonAuriculaire', '#FA', '#flutter', '#TSV', '#tachycardieVentriculaire', '#WPW', '#Brugada']
  → CRT : ['#CRT', '#resynchronisation', '#CRTd', '#CRTp', '#biventriculaire', '#insuffisanceCardiaque']
  → EP : ['#ablation', '#electrophysiologie', '#mapping', '#catheter', '#isolationVeinesPulmonaires', '#ablationFA', '#ablationTV']
  → ICM : ['#moniteurImplantable', '#ICM', '#HolterImplantable', '#syncope', '#monitoringCardiaque']
  → PFA : ['#PFA', '#electroporation', '#pulsedFieldAblation', '#ablationNonThermique', '#FARAPULSE']
  → CSP : ['#LBBAP', '#HisPacing', '#stimulationPhysiologique', '#conductionHis', '#brancheGauche']


##  Fonction verifier_completude() 
Vérifie qu'un post médical contient bien : âge, sexe, antécédents, durée procédure, matériel utilisé.

In [5]:

import re
# ============================================================
#  FONCTION verifier_completude()
# ============================================================

def verifier_completude(texte: str) -> dict:
    """
    Vérifie qu'un post médical contient les informations essentielles.
    Retourne un dict avec le statut de chaque champ et les alertes.
    """
    texte_lower = texte.lower()
    alertes = []
    resultats = {}

    # --- ÂGE ---
    age_patterns = [
        r'\b\d{1,3}\s*ans\b',
        r'\bage\s*:\s*\d+',
        r'\bpatient de \d+',
        r'\b\d{1,3}[-\s]year[s]?[-\s]old\b'
    ]
    age_trouve = any(re.search(p, texte_lower) for p in age_patterns)
    resultats["age"] = age_trouve
    if not age_trouve:
        alertes.append("⚠️ Âge du patient manquant")

    # --- SEXE ---
    sexe_patterns = [
        r'\b(homme|femme|masculin|féminin|male|female)\b',
        r'\b(mr|mme|monsieur|madame)\b',
        r'\b(il|elle) (présente|consulte|est)\b',
        r'\b(patient|patiente)\b'
    ]
    sexe_trouve = any(re.search(p, texte_lower) for p in sexe_patterns)
    resultats["sexe"] = sexe_trouve
    if not sexe_trouve:
        alertes.append("⚠️ Sexe du patient manquant")

    # --- ANTÉCÉDENTS ---
    antecedents_patterns = [
        r'\bantécédents?\b',
        r'\bATCD\b',
        r'\bhistoire médicale\b',
        r'\bdiagnostiqué\b',
        r'\bconnu pour\b',
        r'\bhistory of\b'
    ]
    atcd_trouve = any(re.search(p, texte_lower) for p in antecedents_patterns)
    resultats["antecedents"] = atcd_trouve
    if not atcd_trouve:
        alertes.append("⚠️ Antécédents médicaux manquants")

    # --- DURÉE PROCÉDURE ---
    duree_patterns = [
        r'\b\d+\s*(min|minutes?|heures?|h)\b',
        r'\bdurée\s*:\s*\d+',
        r'\bprocédure de \d+',
        r'\bduring \d+\s*(min|hour)'
    ]
    duree_trouve = any(re.search(p, texte_lower) for p in duree_patterns)
    resultats["duree_procedure"] = duree_trouve
    if not duree_trouve:
        alertes.append("⚠️ Durée de la procédure manquante")

    # --- MATÉRIEL UTILISÉ ---
    materiel_patterns = [
        r'\b(cathéter|sonde|défibrillateur|pacemaker|stimulateur|électrode)\b',
        r'\b(FARAPULSE|PentaRay|ThermoCool|Rhythmia|Carto|EnSite)\b',
        r'\b(système de mapping|console|générateur)\b',
        r'\bmatériel\s*:',
        r'\butilisé\s*:'
    ]
    materiel_trouve = any(re.search(p, texte_lower) for p in materiel_patterns)
    resultats["materiel"] = materiel_trouve
    if not materiel_trouve:
        alertes.append("⚠️ Matériel utilisé non mentionné")

    # --- RÉSUMÉ ---
    complet = len(alertes) == 0
    return {
        "complet": complet,
        "champs": resultats,
        "alertes": alertes
    }


# --- TESTS ---
post_complet = """
Patient de 67 ans, homme, antécédents de fibrillation auriculaire et d'hypertension.
Ablation par radiofréquence réalisée avec cathéter ThermoCool.
Durée de la procédure : 95 minutes. Résultat : succès.
"""

post_incomplet = """
Ablation FA réalisée avec succès. Pas de complications.
"""

print("=" * 50)
print("TEST 1 — Post complet")
print("=" * 50)
r1 = verifier_completude(post_complet)
print(f"Complet : {r1['complet']}")
print(f"Champs  : {r1['champs']}")
if r1['alertes']:
    for a in r1['alertes']: print(a)
else:
    print("✅ Toutes les informations sont présentes !")

print()
print("=" * 50)
print("TEST 2 — Post incomplet")
print("=" * 50)
r2 = verifier_completude(post_incomplet)
print(f"Complet : {r2['complet']}")
for a in r2['alertes']:
    print(a)

TEST 1 — Post complet
Complet : True
Champs  : {'age': True, 'sexe': True, 'antecedents': True, 'duree_procedure': True, 'materiel': True}
✅ Toutes les informations sont présentes !

TEST 2 — Post incomplet
Complet : False
⚠️ Âge du patient manquant
⚠️ Sexe du patient manquant
⚠️ Antécédents médicaux manquants
⚠️ Durée de la procédure manquante
⚠️ Matériel utilisé non mentionné


##  Trends par tag 
Calcule les points par tag selon les réactions : Extremely useful = 3pts, Useful = 1pt.

In [6]:
# ============================================================
#  TRENDS PAR TAG — Partie 2 CDC
#  Extremely useful = 3pts | Useful = 1pt
# ============================================================

def calculer_trends(posts: list) -> dict:
    """
    Calcule les points par tag selon les réactions.
    
    Chaque post doit être un dict avec :
      - 'tags'             : list de tags (ex: ['#ablation', '#FA'])
      - 'extremely_useful' : int (nombre de réactions 'Extremely useful')
      - 'useful'           : int (nombre de réactions 'Useful')
    
    Retourne un dict {tag: points} trié par score décroissant.
    """
    scores = {}

    for post in posts:
        points = (post.get("extremely_useful", 0) * 3) + (post.get("useful", 0) * 1)
        for tag in post.get("tags", []):
            scores[tag] = scores.get(tag, 0) + points

    # Trier par score décroissant
    scores_tries = dict(sorted(scores.items(), key=lambda x: x[1], reverse=True))
    return scores_tries


def afficher_trends(scores: dict, top_n: int = 10):
    """Affiche le classement des tags les plus utiles."""
    print(f"\n🏆 TOP {top_n} TAGS LES PLUS UTILES")
    print("-" * 40)
    for i, (tag, pts) in enumerate(list(scores.items())[:top_n], 1):
        barre = "█" * min(pts, 30)
        print(f"  {i:2}. {tag:<30} {pts:4} pts  {barre}")


# --- DONNÉES DE TEST ---
posts_exemple = [
    {
        "tags": ["#ablation", "#FA", "#electrophysiologie"],
        "extremely_useful": 5,
        "useful": 3
    },
    {
        "tags": ["#pacemaker", "#bradycardie", "#BAV"],
        "extremely_useful": 2,
        "useful": 8
    },
    {
        "tags": ["#ablation", "#PFA", "#electrophysiologie"],
        "extremely_useful": 10,
        "useful": 2
    },
    {
        "tags": ["#CRT", "#resynchronisation", "#insuffisanceCardiaque"],
        "extremely_useful": 1,
        "useful": 4
    },
    {
        "tags": ["#LBBAP", "#stimulationPhysiologique", "#pacemaker"],
        "extremely_useful": 4,
        "useful": 6
    },
    {
        "tags": ["#ICM", "#syncope", "#monitoringCardiaque"],
        "extremely_useful": 0,
        "useful": 5
    }
]

scores = calculer_trends(posts_exemple)
afficher_trends(scores, top_n=10)


🏆 TOP 10 TAGS LES PLUS UTILES
----------------------------------------
   1. #ablation                        50 pts  ██████████████████████████████
   2. #electrophysiologie              50 pts  ██████████████████████████████
   3. #pacemaker                       32 pts  ██████████████████████████████
   4. #PFA                             32 pts  ██████████████████████████████
   5. #FA                              18 pts  ██████████████████
   6. #LBBAP                           18 pts  ██████████████████
   7. #stimulationPhysiologique        18 pts  ██████████████████
   8. #bradycardie                     14 pts  ██████████████
   9. #BAV                             14 pts  ██████████████
  10. #CRT                              7 pts  ███████
